# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## My Week 02 Solution

## Import Required Libraries

In this cell, we import all the required standard Python libraries and third-party packages needed for the notebook. These imports support environment handling, API communication, type hinting, temporary file handling, subprocess execution, and the Gradio interface.

In [ ]:
# Standard library imports
import os
import json
import sys
import subprocess
import pydoc
import tempfile

# Third-party imports
import gradio as gr

from typing import Dict, List, Any, Tuple
from dotenv import load_dotenv
from openai import OpenAI

# Optional: useful for debugging package version issues
# print(f"Gradio version: {gr.__version__}")

## Load Environment Variables and Configure Model Providers

In this cell, we load API credentials from environment variables and initialize clients for different model providers such as OpenAI, OpenRouter, and Ollama.

The code also checks whether Ollama is available locally, builds a catalog of supported models, and creates a helper function to display the provider and model status in markdown format. This makes the notebook flexible by allowing it to work with cloud-based or local models depending on what is available in the environment.

In [ ]:
## Load the environment variables

# Load variables from the .env file and allow overriding existing values
load_dotenv(override=True)

# Read API keys from environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Create OpenAI client only if the API key is available
openai_client = OpenAI(api_key=openai_api_key) if openai_api_key else None

# Create OpenRouter client only if the API key is available
openrouter_client = (
    OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")
    if openrouter_api_key
    else None
)

# Ollama can be used without a cloud API key if running locally
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


def is_ollama_available() -> bool:
    """
    Check whether the local Ollama server is reachable.

    Returns:
        bool: True if Ollama is available and responding, otherwise False.
    """
    try:
        ollama_client.models.list()
        return True
    except Exception:
        return False


# Store the Ollama availability status for later use
OLLAMA_AVAILABLE = is_ollama_available()


def build_model_catalog() -> Dict[str, Dict[str, Any]]:
    """
    Build a catalog of all available models across supported providers.

    The catalog stores:
    - client: the initialized API client
    - model: model name
    - supports_tools: whether tool calling is supported
    - supports_audio: whether audio output is supported
    - max_tokens: optional response token limit for cost control

    Returns:
        Dict[str, Dict[str, Any]]: Dictionary containing available model configurations.
    """
    catalog = {}

    # Add OpenAI models if OpenAI client is configured
    if openai_client:
        catalog["OpenAI - gpt-4.1-mini"] = {
            "client": openai_client,
            "model": "gpt-4.1-mini",
            "supports_tools": True,
            "supports_audio": True,
        }
        catalog["OpenAI - gpt-4o-mini"] = {
            "client": openai_client,
            "model": "gpt-4o-mini",
            "supports_tools": True,
            "supports_audio": True,
        }

    # Add OpenRouter models if OpenRouter client is configured
    if openrouter_client:
        catalog["OpenRouter - openai/gpt-4o-mini"] = {
            "client": openrouter_client,
            "model": "openai/gpt-4o-mini",
            "supports_tools": True,
            "supports_audio": False,
            # Keep responses within budget for credit-limited OpenRouter keys
            "max_tokens": 1200,
        }

    # Add local Ollama model if the local server is available
    if OLLAMA_AVAILABLE:
        catalog["Ollama - llama3.2"] = {
            "client": ollama_client,
            "model": "llama3.2",
            "supports_tools": False,
            "supports_audio": False,
        }

    return catalog


# Build the complete model catalog and extract the list of model names
MODEL_CATALOG = build_model_catalog()
MODEL_NAMES = list(MODEL_CATALOG.keys())

# Stop execution if no model provider is available
if not MODEL_NAMES:
    raise RuntimeError(
        "No models available. Set OPENAI_API_KEY or OPENROUTER_API_KEY, or start Ollama locally."
    )


def provider_status_markdown() -> str:
    """
    Generate a markdown summary showing provider availability
    and the capabilities of all loaded models.

    Returns:
        str: Markdown-formatted provider and model status report.
    """
    lines = []
    lines.append("### Provider and Model Status")
    lines.append(f"- OPENAI_API_KEY: {'Available' if openai_api_key else 'Missing'}")
    lines.append(f"- OPENROUTER_API_KEY: {'Available' if openrouter_api_key else 'Missing'}")
    lines.append(f"- Ollama reachable: {'Yes' if OLLAMA_AVAILABLE else 'No'}")

    lines.append("\n**Loaded models**")
    for model_name in MODEL_NAMES:
        cfg = MODEL_CATALOG[model_name]
        tool_status = "Yes" if cfg["supports_tools"] else "No"
        audio_status = "Yes" if cfg["supports_audio"] else "No"
        lines.append(f"- {model_name} | tools: {tool_status} | voice out: {audio_status}")

    return "\n".join(lines)


# Default system prompt used to guide the assistant's behavior
DEFAULT_SYSTEM_PROMPT = """
You are a senior technical mentor and software engineer.
Give concise, correct, practical answers to technical questions.
When useful, include short examples and common pitfalls.
If the user asks for code explanation, explain step-by-step and include why it matters.
If you are uncertain, say what is uncertain and how to verify.
Respond in markdown.
""".strip()

# Display current provider and model availability
print(provider_status_markdown())

## Define Helper Tools for Documentation and Audio Processing

In this cell, we define utility functions that extend the assistant’s capabilities.

- `get_python_reference()` fetches local Python documentation for a given symbol using `pydoc`.
- `python_reference_tool` describes this function in tool schema format so it can be invoked by the model.
- `handle_tool_calls()` processes tool calls returned by the model and formats the responses.
- `transcribe_audio()` converts microphone input into text using OpenAI transcription.
- `speak_text()` converts the assistant’s text response into speech and saves it as an MP3 file.

Together, these functions allow the application to support tool calling, audio transcription, and text-to-speech generation.

In [ ]:
def get_python_reference(symbol: str) -> str:
    """
    Return a local Python help snippet for a given symbol using pydoc.

    Args:
        symbol (str): Name of the Python symbol to look up,
                      for example 'json.dumps' or 'collections.Counter'.

    Returns:
        str: A short documentation snippet if found, otherwise
             an informative fallback message.
    """
    symbol = (symbol or "").strip()

    # Return early if no symbol was provided
    if not symbol:
        return "No symbol provided."

    try:
        # Render Python documentation for the given symbol
        doc = pydoc.render_doc(symbol, title="Help on %s")
        lines = doc.splitlines()

        # Limit output to the first 80 lines to keep the response concise
        return "\n".join(lines[:80])
    except Exception:
        # Return a user-friendly message if documentation is unavailable
        return f"No local Python docs found for '{symbol}'."


# Tool definition for exposing local Python reference lookup to the model
python_reference_tool = {
    "type": "function",
    "function": {
        "name": "get_python_reference",
        "description": "Get local Python docs for a symbol such as json.dumps or itertools.groupby.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "Python symbol name, for example: collections.Counter",
                }
            },
            "required": ["symbol"],
            "additionalProperties": False,
        },
    },
}

# List of tools currently available to the assistant
TOOLS = [python_reference_tool]


def handle_tool_calls(tool_calls) -> List[Dict[str, str]]:
    """
    Execute tool calls returned by the model and package their responses.

    Args:
        tool_calls: A list of tool call objects generated by the model.

    Returns:
        List[Dict[str, str]]: A list of tool response messages formatted
        for continuation in the conversation.
    """
    responses = []

    for tool_call in tool_calls:
        name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")

        # Route the tool call to the matching local function
        if name == "get_python_reference":
            result = get_python_reference(arguments.get("symbol", ""))
        else:
            result = f"Unknown tool call: {name}"

        # Append the tool response in chat-compatible format
        responses.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }
        )

    return responses


def transcribe_audio(audio_path: str) -> str:
    """
    Transcribe an audio file into text using OpenAI audio transcription.

    Args:
        audio_path (str): Path to the audio file.

    Returns:
        str: Transcribed text if transcription succeeds,
             otherwise an empty string.
    """
    # Return empty output if no audio path is provided
    if not audio_path:
        return ""

    # Return empty output if OpenAI client is not configured
    if not openai_client:
        return ""

    with open(audio_path, "rb") as audio_file:
        transcript = openai_client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file,
        )

    return transcript.text


def speak_text(text: str) -> str:
    """
    Convert assistant text into speech and save it as an MP3 file.

    Args:
        text (str): Text to convert into speech.

    Returns:
        str: Path to the generated MP3 file, or None if speech generation
             is not possible.
    """
    # Return None if text is empty or OpenAI client is unavailable
    if not text or not openai_client:
        return None

    speech = openai_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text,
    )

    # Save generated speech to a temporary MP3 file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
        tmp.write(speech.content)
        return tmp.name

## Normalize Chat History and Stream Assistant Responses

In this cell, we define helper functions to prepare chat history and generate assistant responses.

- `normalize_history()` converts Gradio chat history into a clean OpenAI-compatible message format.
- `_extract_text()` extracts plain text from different message content formats such as strings, lists, or dictionaries.
- `stream_assistant_reply()` sends the user query and chat history to the selected model, optionally handles tool calls, and streams the assistant’s response incrementally.

This design makes the chatbot more robust because it supports multiple history formats, provider response structures, and fallback behavior when streaming does not return content.

In [ ]:
def normalize_history(history):
    """
    Convert Gradio chat history into OpenAI-style message dictionaries.

    This function supports multiple history formats:
    1. Dictionary style: {"role": ..., "content": ...}
    2. Object style: objects with role/content attributes (for example, gr.ChatMessage)
    3. Tuple/list style: (user_text, assistant_text)

    Args:
        history: Chat history in one of the supported formats.

    Returns:
        list: A cleaned list of messages in OpenAI chat format.
    """
    clean = []

    for item in history or []:
        # Dict style: {"role": ..., "content": ...}
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in ("user", "assistant"):
                text = _extract_text(content)
                if text.strip():
                    clean.append({"role": role, "content": text})
            continue

        # Object style (e.g., gr.ChatMessage): has role/content attributes
        role = getattr(item, "role", None)
        content = getattr(item, "content", None)
        if role in ("user", "assistant"):
            text = _extract_text(content)
            if text.strip():
                clean.append({"role": role, "content": text})
            continue

        # Older tuple style: (user_text, assistant_text)
        if isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            user_text = _extract_text(user_text)
            assistant_text = _extract_text(assistant_text)

            if user_text.strip():
                clean.append({"role": "user", "content": user_text})
            if assistant_text.strip():
                clean.append({"role": "assistant", "content": assistant_text})

    return clean


def _extract_text(content) -> str:
    """
    Extract plain text from different content shapes.

    Supported formats:
    - string
    - list of dictionaries containing text/content
    - dictionary containing text/content

    Args:
        content: Message content in string, list, or dictionary form.

    Returns:
        str: Extracted plain text. Returns an empty string if no text is found.
    """
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                # OpenAI-style rich content parts
                text = item.get("text") or item.get("content")
                if isinstance(text, str):
                    parts.append(text)
        return "".join(parts)

    if isinstance(content, dict):
        text = content.get("text") or content.get("content")
        if isinstance(text, str):
            return text

    return ""


def stream_assistant_reply(
    user_message: str,
    history: List[Dict[str, str]],
    model_choice: str,
    system_prompt: str,
    use_tool: bool,
):
    """
    Stream the assistant response from the selected model.

    This function:
    - prepares the conversation history
    - selects the configured model and provider
    - optionally handles tool calls before producing the final answer
    - streams the model response incrementally
    - falls back to a normal completion call if streaming returns empty content

    Args:
        user_message (str): Latest user input.
        history (List[Dict[str, str]]): Prior chat history.
        model_choice (str): Selected model name from the model catalog.
        system_prompt (str): System prompt that guides model behavior.
        use_tool (bool): Whether tool calling should be enabled.

    Yields:
        str: Partial or complete assistant response as it is streamed.
    """
    cfg = MODEL_CATALOG[model_choice]
    client = cfg["client"]
    model = cfg["model"]
    supports_tools = cfg["supports_tools"]
    max_tokens = cfg.get("max_tokens")

    # Build the message list with system prompt, prior history, and current user input
    messages = [{"role": "system", "content": system_prompt}] + normalize_history(history)
    messages.append({"role": "user", "content": user_message})

    # Shared request options, with optional token cap for credit-limited providers
    request_kwargs = {"model": model, "messages": messages}
    if max_tokens:
        request_kwargs["max_tokens"] = max_tokens

    # If tool calling is enabled and supported, first resolve tool calls,
    # then stream the final assistant answer
    if use_tool and supports_tools:
        response = client.chat.completions.create(**request_kwargs, tools=TOOLS)

        while True:
            choices = getattr(response, "choices", None) or []
            if not choices:
                break

            choice = choices[0]
            if getattr(choice, "finish_reason", None) != "tool_calls":
                break

            tool_message = getattr(choice, "message", None)
            tool_calls = getattr(tool_message, "tool_calls", None) or []
            if not tool_calls:
                break

            tool_responses = handle_tool_calls(tool_calls)

            # Add tool call request and tool output back into the conversation
            messages.append(tool_message)
            messages.extend(tool_responses)

            # Re-run the completion after resolving tools
            response = client.chat.completions.create(**request_kwargs, tools=TOOLS)

        stream = client.chat.completions.create(**request_kwargs, stream=True)
    else:
        # Directly stream the response if tool calling is not used
        stream = client.chat.completions.create(**request_kwargs, stream=True)

    answer = ""

    # Stream partial text chunks from the provider response
    for chunk in stream:
        choices = getattr(chunk, "choices", None) or []
        if not choices:
            continue

        delta_obj = getattr(choices[0], "delta", None)
        delta = _extract_text(getattr(delta_obj, "content", None))
        if not delta:
            continue

        answer += delta
        yield answer

    # Some providers may return empty streaming output
    # In that case, fall back to a normal non-streamed completion
    if not answer.strip():
        final_response = client.chat.completions.create(**request_kwargs)
        final_choices = getattr(final_response, "choices", None) or []
        if final_choices:
            final_msg = getattr(final_choices[0], "message", None)
            final_text = _extract_text(getattr(final_msg, "content", None))
            if final_text:
                yield final_text

## Build the Gradio Interface and Connect the Chat Workflow

In this cell, we define the main user interaction flow and create the Gradio interface for the technical Q&A prototype.

The implementation includes:
- `add_user_message()` to process text and audio input and append it to chat history
- `generate_assistant()` to stream assistant responses and optionally generate spoken output
- `clear_everything()` to reset the application state
- a Gradio UI with model selection, tool toggle, voice toggle, system prompt editing, chat display, text input, audio input, and control buttons

This cell connects all the earlier helper functions into an end-to-end interactive application.

In [ ]:
def add_user_message(text_message, audio_path, history):
    """
    Add the user's message to the chat history.

    This function accepts either text input, audio input, or both.
    If audio is provided, it is transcribed into text and combined with
    the typed message when both are available.

    Args:
        text_message: User-entered text input.
        audio_path: File path of the uploaded or recorded audio.
        history: Existing chatbot conversation history.

    Returns:
        tuple: Resets text input, resets audio input, and returns updated history.

    Raises:
        gr.Error: If no text or audio input is provided, or if transcription
                  is requested without an OpenAI API key.
    """
    history = history or []
    text_message = (text_message or "").strip()

    transcript = ""
    if audio_path:
        if not openai_client:
            raise gr.Error("Audio transcription requires OPENAI_API_KEY.")
        transcript = transcribe_audio(audio_path).strip()

    # Combine typed text and voice transcript when both are available
    if text_message and transcript:
        user_text = f"{text_message}\n\n(Voice transcript) {transcript}"
    elif text_message:
        user_text = text_message
    elif transcript:
        user_text = transcript
    else:
        raise gr.Error("Provide text or audio input.")

    # Add the new user message in the format expected by the Gradio Chatbot
    history = history + [{"role": "user", "content": user_text}]
    return "", None, history


def generate_assistant(history, model_choice, system_prompt, use_tool, voice_reply):
    """
    Generate and stream the assistant response.

    This function:
    - reads the most recent user message from chat history
    - sends the request to the selected model
    - streams the assistant reply incrementally
    - optionally converts the final response to speech

    Args:
        history: Current chatbot conversation history.
        model_choice: Selected model from the available model catalog.
        system_prompt: System instruction that guides the assistant behavior.
        use_tool: Whether tool calling is enabled.
        voice_reply: Whether the final assistant response should also be spoken.

    Yields:
        tuple: Updated chatbot history and optional audio output path.
    """
    history = history or []

    # If there is no history, return without generating anything
    if not history:
        yield history, None
        return

    # Support both dictionary-based and object-based chat history items
    last = history[-1]
    if isinstance(last, dict):
        user_message = _extract_text(last.get("content"))
    else:
        user_message = _extract_text(getattr(last, "content", ""))

    # Handle empty user messages gracefully
    if not user_message.strip():
        yield history + [{"role": "assistant", "content": "Please enter a message."}], None
        return

    prior_history = history[:-1]
    final_text = ""

    try:
        # Stream the assistant response incrementally
        for partial in stream_assistant_reply(
            user_message=user_message,
            history=prior_history,
            model_choice=model_choice,
            system_prompt=system_prompt,
            use_tool=use_tool,
        ):
            final_text = partial
            yield prior_history + [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": partial},
            ], None

    except Exception as e:
        # Return a readable error message inside the chat if generation fails
        error_msg = f"Error while generating response: {e}"
        yield prior_history + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": error_msg},
        ], None
        return

    # Fallback message if no response text was returned by the model
    if not final_text.strip():
        final_text = (
            "No response text returned by the selected model. "
            "Try another model or disable tools for this request."
        )

    # Optionally generate speech output for the final assistant response
    audio_output = speak_text(final_text) if voice_reply else None

    yield prior_history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": final_text},
    ], audio_output


def clear_everything():
    """
    Reset the chatbot state and all user input/output components.

    Returns:
        tuple: Empty chatbot history, cleared text input, cleared audio input,
               and cleared audio output.
    """
    return [], "", None, None


with gr.Blocks(title="Technical Q/A Prototyper") as demo:
    # Application title and short feature summary
    gr.Markdown("# Technical Q/A Prototyper\nStreaming + model switch + expert prompt + tool calling + optional voice")

    # Display current provider/model availability
    status_md = gr.Markdown(value=provider_status_markdown())
    refresh_status = gr.Button("Refresh status")

    with gr.Row():
        model_choice = gr.Dropdown(
            choices=MODEL_NAMES,
            value=MODEL_NAMES[0],
            label="Model",
        )
        use_tool = gr.Checkbox(value=True, label="Enable Python reference tool")
        voice_reply = gr.Checkbox(value=False, label="Speak replies (OpenAI key required)")

    # Editable system prompt to control assistant behavior
    system_prompt = gr.Textbox(
        label="System prompt (expertise)",
        value=DEFAULT_SYSTEM_PROMPT,
        lines=6,
    )

    # Chatbot configured to use messages-format data
    chatbot = gr.Chatbot(height=450, label="Technical Assistant")

    with gr.Row():
        text_input = gr.Textbox(label="Ask a technical question", lines=3)

        # Accept both live microphone input and uploaded audio files
        audio_input = gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="Voice input (microphone or upload)",
        )

    # User guidance for microphone issues
    gr.Markdown(
        "If your browser says no microphone found, allow mic permissions or upload an audio file instead."
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        cancel = gr.Button("Cancel generation", variant="stop")
        clear = gr.Button("Clear")

    # Audio player for assistant voice responses
    audio_output = gr.Audio(label="Assistant voice reply", autoplay=True)

    # Store event references so in-progress generation can be cancelled
    send_event = send.click(
        add_user_message,
        inputs=[text_input, audio_input, chatbot],
        outputs=[text_input, audio_input, chatbot],
    ).then(
        generate_assistant,
        inputs=[chatbot, model_choice, system_prompt, use_tool, voice_reply],
        outputs=[chatbot, audio_output],
    )

    submit_event = text_input.submit(
        add_user_message,
        inputs=[text_input, audio_input, chatbot],
        outputs=[text_input, audio_input, chatbot],
    ).then(
        generate_assistant,
        inputs=[chatbot, model_choice, system_prompt, use_tool, voice_reply],
        outputs=[chatbot, audio_output],
    )

    # Stop currently running generation tasks
    cancel.click(
        fn=None,
        inputs=None,
        outputs=None,
        cancels=[send_event, submit_event],
    )

    # Clear the full application state
    clear.click(
        clear_everything,
        inputs=[],
        outputs=[chatbot, text_input, audio_input, audio_output],
    )

    # Refresh provider/model status display
    refresh_status.click(
        provider_status_markdown,
        inputs=[],
        outputs=[status_md],
    )

print("UI object created. Run the next cell to launch it.")

## Launch the Application

In this final cell, we launch the Gradio-based Technical Q/A Prototyper interface.

Setting `inbrowser=True` automatically opens the application in the default web browser, making it easy to interact with the chatbot UI for testing text input, audio input, streaming responses, tool usage, and optional voice output.

In [ ]:
# Launch the Gradio application in the browser
demo.launch(inbrowser=True)